# LexBanglaRAG: Qwen2.5-14B No-RAG Frozen Production Pipeline

**Scientific Purpose:** Establish the rigorous, un-retrieved parametric baseline for Qwen2.5-14B on the exact 180 expert-verified Bangladeshi legal QA benchmark.

### Hardware & Environment Specifications:
- **Accelerator:** GPU T4 x 1 (Single NVIDIA T4 GPU, 16 GB VRAM)
- **Internet:** ON (for downloading HuggingFace weights)
- **Dataset Attached:** Dataset containing `gold_qa_set_v2.csv`

In [1]:
# Cell 1: Install Required Libraries
!pip install -q "bitsandbytes>=0.43.0" "accelerate>=1.0.0"
print("✓ Dependencies installed successfully.")

✓ Dependencies installed successfully.


In [2]:
# Cell 2: Locate Dataset & Verify Integrity (180 Questions, 49 Acts, Zero Alias Collision)
import os, sys, glob, re, time, unicodedata
from collections import Counter
import pandas as pd
import torch

def find_input(filename):
    candidates = [f"/kaggle/input/**/{filename}", f"/kaggle/working/{filename}", filename]
    found = []
    for p in candidates:
        found.extend(glob.glob(p, recursive=True))
    found = list(dict.fromkeys(p for p in found if os.path.isfile(p)))
    if not found:
        raise FileNotFoundError(f"Missing {filename}! Please attach your dataset in Kaggle.")
    found.sort(key=lambda p: (not p.startswith("/kaggle/input/"), len(p)))
    return found[0]

QA_PATH = find_input("gold_qa_set_v2.csv")
print(f"Loading gold benchmark from: {QA_PATH}")
df_gold = pd.read_csv(QA_PATH, encoding="utf-8-sig")

# A. Null check BEFORE string conversion
assert df_gold["qa_id"].notna().all(), "FATAL ERROR: Null qa_id found in gold dataset prior to conversion!"
# B. Explicit string conversion AND whitespace stripping assigned back to df_gold
df_gold["qa_id"] = df_gold["qa_id"].astype(str).str.strip()
# C. Blank / whitespace rejection AFTER string conversion
assert (df_gold["qa_id"] != "").all(), "FATAL ERROR: Blank/whitespace qa_id found after conversion!"
# D. Uniqueness and count assertions
assert df_gold["qa_id"].is_unique, "FATAL ERROR: Duplicate qa_id found in gold dataset!"
assert len(df_gold) == 180, f"FATAL ERROR: Dataset has {len(df_gold)} rows, expected exactly 180!"
expected_ids = set(df_gold["qa_id"])
print(f"✓ Dataset Integrity Verified: Exactly 180 unique, non-null, non-blank qa_ids loaded.")

CANONICAL_ACT_ALIASES = {   'অপরাধ সম্পর্কিত বিষয়ে পারস্পরিক সহায়তা আইন, ২০১২': [   'অপরাধ সম্পর্কিত বিষয়ে পারস্পরিক সহায়তা আইন',
                                                            'অপরাধ সম্পর্কিত বিষয়ে পারস্পরিক সহায়তা আইন 2012'],
    'অর্পিত সম্পত্তি প্রত্যর্পণ আইন, ২০০১': ['অর্পিত সম্পত্তি প্রত্যর্পণ আইন', 'অর্পিত সম্পত্তি প্রত্যর্পণ আইন 2001'],
    'অস্থাবর সম্পত্তি হুকুমদখল আইন, ১৯৮৮': ['অস্থাবর সম্পত্তি হুকুমদখল আইন', 'অস্থাবর সম্পত্তি হুকুমদখল আইন 1988'],
    'আইন-শৃঙ্খলা বিঘ্নকারী অপরাধ (দ্রুত বিচার) (সংশোধন) আইন, ২০২৪': [   'আইন শৃঙ্খলা বিঘ্নকারী অপরাধ দ্রুত বিচার '
                                                                        'সংশোধন আইন',
                                                                        'আইন শৃঙ্খলা বিঘ্নকারী অপরাধ দ্রুত বিচার '
                                                                        'সংশোধন আইন 2024'],
    'আইন-শৃঙ্খলা বিঘ্নকারী অপরাধ (দ্রুত বিচার) আইন, ২০০২': [   'আইন শৃঙ্খলা বিঘ্নকারী অপরাধ দ্রুত বিচার আইন',
                                                               'আইন শৃঙ্খলা বিঘ্নকারী অপরাধ দ্রুত বিচার আইন 2002'],
    'আভ্যন্তরীণ নৌযান শ্রমিক (নিয়োগ-নিয়ন্ত্রণ) আইন, ১৯৯২': [   'আভ্যন্তরীণ নৌযান শ্রমিক নিয়োগ নিয়ন্ত্রণ আইন',
                                                               'আভ্যন্তরীণ নৌযান শ্রমিক নিয়োগ নিয়ন্ত্রণ আইন 1992'],
    'এসিড অপরাধ দমন আইন, ২০০২': ['এসিড অপরাধ দমন আইন', 'এসিড অপরাধ দমন আইন 2002'],
    'ওয়াক্\u200cফ (সম্পত্তি হস্তান্তর ও উন্নয়ন) বিশেষ বিধান আইন, ২০১৩': [   'ওয়াক্\u200cফ সম্পত্তি হস্তান্তর ও '
                                                                            'উন্নয়ন বিশেষ বিধান আইন',
                                                                            'ওয়াক্\u200cফ সম্পত্তি হস্তান্তর ও '
                                                                            'উন্নয়ন বিশেষ বিধান আইন 2013'],
    'কারাগারে আটক সাজাপ্রাপ্ত নারীদের বিশেষ সুবিধা আইন, ২০০৬': [   'কারাগারে আটক সাজাপ্রাপ্ত নারীদের বিশেষ সুবিধা আইন',
                                                                   'কারাগারে আটক সাজাপ্রাপ্ত নারীদের বিশেষ সুবিধা আইন '
                                                                   '2006'],
    'গণকর্মচারী (বিদেশি নাগরিকের সহিত বিবাহ) আইন, ২০১৫': [   'গণকর্মচারী বিদেশি নাগরিকের সহিত বিবাহ আইন',
                                                             'গণকর্মচারী বিদেশি নাগরিকের সহিত বিবাহ আইন 2015'],
    'গান্ধী আশ্রম (ট্রাস্টি বোর্ড) আইন, ২০২১': [   'গান্ধী আশ্রম ট্রাস্টি বোর্ড আইন',
                                                   'গান্ধী আশ্রম ট্রাস্টি বোর্ড আইন 2021'],
    'ঢাকা এলিভেটেড এক্সপ্রেসওয়ে প্রকল্প (ভূমি অধিগ্রহণ) আইন , ২০১১': [   'ঢাকা এলিভেটেড এক্সপ্রেসওয়ে প্রকল্প ভূমি '
                                                                         'অধিগ্রহণ আইন',
                                                                         'ঢাকা এলিভেটেড এক্সপ্রেসওয়ে প্রকল্প ভূমি '
                                                                         'অধিগ্রহণ আইন 2011'],
    'নারী ও শিশু নির্যাতন দমন (সংশোধন) অধ্যাদেশ, ২০২৫': [   'নারী ও শিশু নির্যাতন দমন সংশোধন অধ্যাদেশ',
                                                            'নারী ও শিশু নির্যাতন দমন সংশোধন অধ্যাদেশ 2025'],
    'নারী ও শিশু নির্যাতন দমন (সংশোধন) আইন, ২০২০': [   'নারী ও শিশু নির্যাতন দমন সংশোধন আইন',
                                                       'নারী ও শিশু নির্যাতন দমন সংশোধন আইন 2020'],
    'নারী ও শিশু নির্যাতন দমন আইন, ২০০০': ['নারী ও শিশু নির্যাতন দমন আইন', 'নারী ও শিশু নির্যাতন দমন আইন 2000'],
    'নির্যাতন এবং হেফাজতে মৃত্যু (নিবারণ) আইন, ২০১৩': [   'নির্যাতন এবং হেফাজতে মৃত্যু নিবারণ আইন',
                                                          'নির্যাতন এবং হেফাজতে মৃত্যু নিবারণ আইন 2013'],
    'পণ্য উৎপাদনশীল রাষ্ট্রায়ত্ত শিল্প প্রতিষ্ঠান শ্রমিক (চাকরির শর্তাবলী) আইন, ২০১৮': [   'পণ্য উৎপাদনশীল '
                                                                                           'রাষ্ট্রায়ত্ত শিল্প '
                                                                                           'প্রতিষ্ঠান শ্রমিক চাকরির '
                                                                                           'শর্তাবলী আইন',
                                                                                           'পণ্য উৎপাদনশীল '
                                                                                           'রাষ্ট্রায়ত্ত শিল্প '
                                                                                           'প্রতিষ্ঠান শ্রমিক চাকরির '
                                                                                           'শর্তাবলী আইন 2018'],
    'পদ্মা বহুমুখী সেতু প্রকল্প (ভূমি অধিগ্রহণ) আইন, ২০০৯': [   'পদ্মা বহুমুখী সেতু প্রকল্প ভূমি অধিগ্রহণ আইন',
                                                                'পদ্মা বহুমুখী সেতু প্রকল্প ভূমি অধিগ্রহণ আইন 2009'],
    'পারিবারিক আদালত আইন, ২০২৩': ['পারিবারিক আদালত আইন', 'পারিবারিক আদালত আইন 2023'],
    'পারিবারিক সহিংসতা (প্রতিরোধ ও সুরক্ষা) আইন, ২০১০': [   'পারিবারিক সহিংসতা প্রতিরোধ ও সুরক্ষা আইন',
                                                            'পারিবারিক সহিংসতা প্রতিরোধ ও সুরক্ষা আইন 2010'],
    'পার্বত্য চট্টগ্রাম ভূমি-বিরোধ নিষ্পত্তি কমিশন আইন, ২০০১': [   'পার্বত্য চট্টগ্রাম ভূমি বিরোধ নিষ্পত্তি কমিশন আইন',
                                                                   'পার্বত্য চট্টগ্রাম ভূমি বিরোধ নিষ্পত্তি কমিশন আইন '
                                                                   '2001'],
    'পায়রা বন্দর প্রকল্প (ভূমি অধিগ্রহণ) আইন, ২০১৬': [   'পায়রা বন্দর প্রকল্প ভূমি অধিগ্রহণ আইন',
                                                         'পায়রা বন্দর প্রকল্প ভূমি অধিগ্রহণ আইন 2016'],
    'বাংলাদেশ ইপিজেড শ্রম আইন, ২০১৯': ['বাংলাদেশ ইপিজেড শ্রম আইন', 'বাংলাদেশ ইপিজেড শ্রম আইন 2019'],
    'বাংলাদেশ চা শ্রমিক কল্যাণ তহবিল আইন, ২০১৬': [   'বাংলাদেশ চা শ্রমিক কল্যাণ তহবিল আইন',
                                                     'বাংলাদেশ চা শ্রমিক কল্যাণ তহবিল আইন 2016'],
    'বাংলাদেশ শিশু একাডেমি আইন, ২০১৮': ['বাংলাদেশ শিশু একাডেমি আইন', 'বাংলাদেশ শিশু একাডেমি আইন 2018'],
    'বাংলাদেশ শিশু হাসপাতাল ও ইনস্টিটিউট আইন, ২০২১': [   'বাংলাদেশ শিশু হাসপাতাল ও ইনস্টিটিউট আইন',
                                                         'বাংলাদেশ শিশু হাসপাতাল ও ইনস্টিটিউট আইন 2021'],
    'বাংলাদেশ শ্রম আইন, ২০০৬': ['বাংলাদেশ শ্রম আইন', 'বাংলাদেশ শ্রম আইন 2006'],
    'বাংলাদেশ শ্রমিক কল্যাণ ফাউন্ডেশন আইন, ২০০৬': [   'বাংলাদেশ শ্রমিক কল্যাণ ফাউন্ডেশন আইন',
                                                      'বাংলাদেশ শ্রমিক কল্যাণ ফাউন্ডেশন আইন 2006'],
    'বাল্যবিবাহ নিরোধ আইন, ২০১৭': ['বাল্যবিবাহ নিরোধ আইন', 'বাল্যবিবাহ নিরোধ আইন 2017'],
    'বিমান-নিরাপত্তা বিরোধী অপরাধ দমন আইন, ১৯৯৭': [   'বিমান নিরাপত্তা বিরোধী অপরাধ দমন আইন',
                                                      'বিমান নিরাপত্তা বিরোধী অপরাধ দমন আইন 1997'],
    'ব্যক্তি মালিকানাধীন সড়ক পরিবহন শ্রমিক কল্যাণ তহবিল': ['ব্যক্তি মালিকানাধীন সড়ক পরিবহন শ্রমিক কল্যাণ তহবিল'],
    'ভূমি অপরাধ প্রতিরোধ ও প্রতিকার আইন, ২০২৩': [   'ভূমি অপরাধ প্রতিরোধ ও প্রতিকার আইন',
                                                    'ভূমি অপরাধ প্রতিরোধ ও প্রতিকার আইন 2023'],
    'ভূমি আপীল বোর্ড আইন, ১৯৮৯': ['ভূমি আপীল বোর্ড আইন', 'ভূমি আপীল বোর্ড আইন 1989'],
    'ভূমি উন্নয়ন কর আইন, ২০২৩': ['ভূমি উন্নয়ন কর আইন', 'ভূমি উন্নয়ন কর আইন 2023'],
    'ভূমি সংস্কার আইন, ২০২৩': ['ভূমি সংস্কার আইন', 'ভূমি সংস্কার আইন 2023'],
    'ভূমি সংস্কার বোর্ড আইন, ১৯৮৯': ['ভূমি সংস্কার বোর্ড আইন', 'ভূমি সংস্কার বোর্ড আইন 1989'],
    'ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ, ১৯৮৪': [   'ভূমি খতিয়ান পার্বত্য চট্টগ্রাম অধ্যাদেশ',
                                                           'ভূমি খতিয়ান পার্বত্য চট্টগ্রাম অধ্যাদেশ 1984'],
    'মহা হিসাব নিরীক্ষক ও নিয়ন্ত্রক (পারিশ্রমিক ও বিশেষ অধিকার) আইন, ১৯৯৮': [   'মহা হিসাব নিরীক্ষক ও নিয়ন্ত্রক '
                                                                                'পারিশ্রমিক ও বিশেষ অধিকার আইন',
                                                                                'মহা হিসাব নিরীক্ষক ও নিয়ন্ত্রক '
                                                                                'পারিশ্রমিক ও বিশেষ অধিকার আইন 1998'],
    'মাতৃদুগ্ধ বিকল্প, শিশু খাদ্য, বাণিজ্যিকভাবে প্রস্ত্ততকৃত শিশুর বাড়তি খাদ্য ও উহা ব্যবহারের সরঞ্জামাদি (বিপণন নিয়ন্ত্রণ) আইন, ২০১৩': [   'মাতৃদুগ্ধ '
                                                                                                                                             'বিকল্প '
                                                                                                                                             'শিশু '
                                                                                                                                             'খাদ্য '
                                                                                                                                             'বাণিজ্যিকভাবে '
                                                                                                                                             'প্রস্ত্ততকৃত '
                                                                                                                                             'শিশুর '
                                                                                                                                             'বাড়তি '
                                                                                                                                             'খাদ্য '
                                                                                                                                             'ও '
                                                                                                                                             'উহা '
                                                                                                                                             'ব্যবহারের '
                                                                                                                                             'সরঞ্জামাদি '
                                                                                                                                             'বিপণন '
                                                                                                                                             'নিয়ন্ত্রণ '
                                                                                                                                             'আইন',
                                                                                                                                             'মাতৃদুগ্ধ '
                                                                                                                                             'বিকল্প '
                                                                                                                                             'শিশু '
                                                                                                                                             'খাদ্য '
                                                                                                                                             'বাণিজ্যিকভাবে '
                                                                                                                                             'প্রস্ত্ততকৃত '
                                                                                                                                             'শিশুর '
                                                                                                                                             'বাড়তি '
                                                                                                                                             'খাদ্য '
                                                                                                                                             'ও '
                                                                                                                                             'উহা '
                                                                                                                                             'ব্যবহারের '
                                                                                                                                             'সরঞ্জামাদি '
                                                                                                                                             'বিপণন '
                                                                                                                                             'নিয়ন্ত্রণ '
                                                                                                                                             'আইন '
                                                                                                                                             '2013'],
    'যমুনা বহুমুখী সেতু প্রকল্প (ভূমি অধিগ্রহণ) আইন, ১৯৯৫': [   'যমুনা বহুমুখী সেতু প্রকল্প ভূমি অধিগ্রহণ আইন',
                                                                'যমুনা বহুমুখী সেতু প্রকল্প ভূমি অধিগ্রহণ আইন 1995'],
    'যৌতুক নিরোধ আইন, ২০১৮': ['যৌতুক নিরোধ আইন', 'যৌতুক নিরোধ আইন 2018'],
    'রেলওয়ে সম্পত্তি (অবৈধ দখল উদ্ধার) আইন, ২০১৬': [   'রেলওয়ে সম্পত্তি অবৈধ দখল উদ্ধার আইন',
                                                       'রেলওয়ে সম্পত্তি অবৈধ দখল উদ্ধার আইন 2016'],
    'শিশু আইন, ২০১৩': ['শিশু আইন 2013'],
    'শিশু দিবাযত্ন কেন্দ্র আইন, ২০২১': ['শিশু দিবাযত্ন কেন্দ্র আইন', 'শিশু দিবাযত্ন কেন্দ্র আইন 2021'],
    'শিশু-মাতৃ স্বাস্থ্য ইনস্\u200cটিটিউট আইন, ২০০২': [   'শিশু মাতৃ স্বাস্থ্য ইনস্\u200cটিটিউট আইন',
                                                          'শিশু মাতৃ স্বাস্থ্য ইনস্\u200cটিটিউট আইন 2002'],
    'সন্ত্রাসমূলক অপরাধ দমন (বিশেষ বিধান) আইন, ১৯৯৪': [   'সন্ত্রাসমূলক অপরাধ দমন বিশেষ বিধান আইন',
                                                          'সন্ত্রাসমূলক অপরাধ দমন বিশেষ বিধান আইন 1994'],
    'সুরক্ষিত লেনদেন (অস্থাবর সম্পত্তি) আইন, ২০২৩': [   'সুরক্ষিত লেনদেন অস্থাবর সম্পত্তি আইন',
                                                        'সুরক্ষিত লেনদেন অস্থাবর সম্পত্তি আইন 2023'],
    'স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন, ২০১৭': [   'স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন',
                                                         'স্থাবর সম্পত্তি অধিগ্রহণ ও হুকুমদখল আইন 2017'],
    'হিন্দু বিবাহ নিবন্ধন আইন, ২০১২': ['হিন্দু বিবাহ নিবন্ধন আইন', 'হিন্দু বিবাহ নিবন্ধন আইন 2012']}

# Exact Set Equality between Gold Acts and Canonical Dictionary Keys
gold_act_set = set(df_gold["act_title"].dropna().str.strip().unique())
dict_act_set = set(CANONICAL_ACT_ALIASES.keys())
assert gold_act_set == dict_act_set, f"FATAL: Act set mismatch! Diff (gold - dict): {gold_act_set - dict_act_set}"
assert len(dict_act_set) == 49, f"FATAL: Expected exactly 49 Acts, found {len(dict_act_set)}"

# Explicit Zero-Collision Alias Ownership Verification
alias_to_act_map = {}
for act_canonical, alias_list in CANONICAL_ACT_ALIASES.items():
    assert len(alias_list) > 0, f"FATAL: Act {act_canonical} has no aliases!"
    for a in alias_list:
        norm_a = unicodedata.normalize("NFC", str(a)).strip().lower()
        norm_a = re.sub(r"\s+", " ", norm_a)
        assert norm_a not in alias_to_act_map, f"FATAL COLLISION: Normalized alias '{norm_a}' belongs to BOTH '{alias_to_act_map[norm_a]}' AND '{act_canonical}'!"
        alias_to_act_map[norm_a] = act_canonical

print(f"✓ Canonical Alias Dictionary Verified: Exact set equality (49 == 49) and PROVEN ZERO COLLISIONS across all {len(alias_to_act_map)} aliases.")

Loading gold benchmark from: /kaggle/input/datasets/tanmimalamnuhin/gold-qa-v2/gold_qa_set_v2.csv
✓ Dataset Integrity Verified: Exactly 180 unique, non-null, non-blank qa_ids loaded.
✓ Canonical Alias Dictionary Verified: Exact set equality (49 == 49) and PROVEN ZERO COLLISIONS across all 96 aliases.


In [4]:
# Cell 3: Load Qwen2.5-14B in 4-bit NF4 & Verify Strict cuda:0 Placement
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "FATAL ERROR: CUDA GPU is not available on this Kaggle session!"
assert torch.cuda.current_device() == 0, f"FATAL ERROR: Active CUDA device is {torch.cuda.current_device()}, expected 0!"

MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Reuse model if already loaded in memory to save 2 minutes
if "model" in locals() and hasattr(model, "generate") and "tokenizer" in locals():
    print("✓ Model is already loaded in memory! Verifying placement...")
else:
    print(f"Loading {MODEL_NAME} in 4-bit NF4 strictly on Kaggle T4 GPU 0 (cuda:0)...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0}
    )
    model.eval()
    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

# Robust verification of placement strictly on cuda:0
print("Verifying layer placement strictly on cuda:0...")
if hasattr(model, "hf_device_map") and model.hf_device_map:
    for layer_name, dev in model.hf_device_map.items():
        assert dev in [0, "0", "cuda:0"], f"FATAL: Layer {layer_name} mapped off GPU 0: {dev}"
else:
    assert model.device.type == "cuda" and (model.device.index == 0 or model.device.index is None), f"FATAL: Model device is {model.device}, expected cuda:0!"
    param_devices = {p.device for p in model.parameters()}
    for p_dev in param_devices:
        assert p_dev.type == "cuda" and (p_dev.index == 0 or p_dev.index is None), f"FATAL: Parameter mapped to {p_dev}, expected cuda:0!"

initial_vram = torch.cuda.memory_allocated(0)/(1024**3)
print(f"✓ Strict Single-GPU Placement Verified (cuda:0 only)! Initial Allocated VRAM: {initial_vram:.2f} GB")

✓ Model is already loaded in memory! Verifying placement...
Verifying layer placement strictly on cuda:0...
✓ Strict Single-GPU Placement Verified (cuda:0 only)! Initial Allocated VRAM: 9.28 GB


In [5]:
# Cell 4: Exact Matched No-RAG Prompt & Generation Function (Stage 6 Cell 8 Match)
SYSTEM_MSG = "তুমি বাংলাদেশের আইনের বিশেষজ্ঞ। প্রশ্নের নির্ভুল আইনি ব্যাখ্যা ও সংশ্লিষ্ট ধারা উল্লেখ করো।"

def make_norag_prompt(question):
    return f"""তুমি বাংলাদেশের আইন সংক্রান্ত একজন বিশেষজ্ঞ। নিচের প্রশ্নটির নির্ভুল আইনি উত্তর দাও।
উত্তরে অবশ্যই প্রযোজ্য আইনের নাম এবং সংশ্লিষ্ট ধারা উল্লেখ করবে। কোনো মনগড়া বা ভুল ধারা উল্লেখ করবে না।

প্রশ্ন:
{question}

আইনি উত্তর (আইন ও ধারাসহ):"""

def generate_norag_answer(question, max_new_tokens=384):
    user_content = make_norag_prompt(str(question).strip())
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_content}
    ]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokenizer.truncation_side = "left"
    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=3800)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()

print("✓ Matched No-RAG Generation Engine Initialized.")

✓ Matched No-RAG Generation Engine Initialized.


In [6]:
# Cell 5: Step 1 — Pre-Flight Engineering Smoke Test (3 Pilot Questions)
# Verifies GPU memory safety (<14.5 GB), token limits, and no runtime crashes
print("="*70)
print("EXECUTING PRE-FLIGHT ENGINEERING SMOKE TEST (3 PILOT QUESTIONS)...")
print("="*70)
torch.cuda.reset_peak_memory_stats(0)

smoke_success = True
for s_idx, (_, s_row) in enumerate(df_gold.head(3).iterrows(), 1):
    t0 = time.time()
    s_ans = generate_norag_answer(s_row["question"])
    t_cost = time.time() - t0
    if not s_ans or len(s_ans.strip()) < 5:
        smoke_success = False
        print(f"FAILED on smoke test item {s_idx}!")
    else:
        print(f"Smoke Item {s_idx}/3 (qa_id: {s_row['qa_id']}) completed in {t_cost:.1f}s")
        print(f"Preview: {s_ans[:90]}...\n")

peak_alloc = torch.cuda.max_memory_allocated(0)/(1024**3)
peak_resv = torch.cuda.max_memory_reserved(0)/(1024**3)

print(f"Peak Allocated VRAM: {peak_alloc:.2f} GB | Peak Reserved VRAM: {peak_resv:.2f} GB / 15.0 GB usable")
assert smoke_success, "FATAL ERROR: Smoke test generation failed!"
assert peak_resv < 14.5, "FATAL ERROR: Peak reserved VRAM dangerously close to OOM limit!"
print("✓ PRE-FLIGHT ENGINEERING SMOKE TEST PASSED SUCCESSFULLY!")
print("Hardware is safe and fully capable of full 180-question execution.")
print("="*70)

EXECUTING PRE-FLIGHT ENGINEERING SMOKE TEST (3 PILOT QUESTIONS)...
Smoke Item 1/3 (qa_id: qa_0001) completed in 46.4s
Preview: পাহাড়ি এলাকার জমির খতিয়ানে যদি ভুল বা ধোঁকাবাজি করে তথ্য ঢুকানো হয়, তাহলে খতিয়ান চূড়া...

Smoke Item 2/3 (qa_id: qa_0002) completed in 43.1s
Preview: বাংলাদেশে, সরকার কোনো কাজের জন্য হঠাৎ করে আপনার জিনিসপত্র নিতে পারে না। যদি সরকার কোনো জিন...

Smoke Item 3/3 (qa_id: qa_0003) completed in 42.7s
Preview: ভূমি সংস্কার বোর্ড বাংলাদেশের ভূমি সংস্কার ও বিস্তার আইন, ২০১৩ এর অধীনে গঠিত হয়। এই আইনের...

Peak Allocated VRAM: 9.57 GB | Peak Reserved VRAM: 9.73 GB / 15.0 GB usable
✓ PRE-FLIGHT ENGINEERING SMOKE TEST PASSED SUCCESSFULLY!
Hardware is safe and fully capable of full 180-question execution.


In [7]:
# Cell 6: Step 2 — Checkpointed 180-Question Inference
CHECKPOINT_PATH = "/kaggle/working/qwen14b_norag_strict_checkpoint.csv"
OUTPUT_PATH = "/kaggle/working/qwen14b_norag_strict_results_180.csv"

completed_records = []
done_ids = set()

if os.path.exists(CHECKPOINT_PATH):
    chk_df = pd.read_csv(CHECKPOINT_PATH, encoding="utf-8-sig")
    assert chk_df["qa_id"].notna().all(), "FATAL ERROR: Null qa_id found in checkpoint!"
    chk_df["qa_id"] = chk_df["qa_id"].astype(str).str.strip()
    assert (chk_df["qa_id"] != "").all(), "FATAL ERROR: Blank/whitespace qa_id found in checkpoint!"
    assert chk_df["qa_id"].is_unique, "FATAL ERROR: Duplicate qa_id found in checkpoint!"
    checkpoint_ids = set(chk_df["qa_id"])
    assert checkpoint_ids.issubset(expected_ids), "FATAL ERROR: Unexpected qa_id found in checkpoint!"
    done_ids = checkpoint_ids
    completed_records = chk_df.to_dict("records")
    print(f"✓ Resuming safely: {len(done_ids)} questions already processed in checkpoint.")

remaining_df = df_gold[~df_gold["qa_id"].isin(done_ids)].copy()
print(f"Starting batch generation for remaining {len(remaining_df)}/180 questions...")

start_time = time.time()
for idx, (_, row) in enumerate(remaining_df.iterrows(), start=len(done_ids)+1):
    q_id = row["qa_id"]
    q_text = row["question"]
    gold_act = row["act_title"]
    gold_sec = str(row["section_number"])
    gold_ans = row["final_answer"]
    
    gen_ans = generate_norag_answer(q_text)
    
    completed_records.append({
        "qa_id": q_id,
        "question": q_text,
        "act_title": gold_act,
        "section_number": gold_sec,
        "final_answer": gold_ans,
        "qwen14b_norag_answer": gen_ans
    })
    
    if idx % 10 == 0 or idx == 180:
        pd.DataFrame(completed_records).to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
        elapsed = time.time() - start_time
        print(f"[{idx}/180] Processed ({q_id}) | Elapsed: {elapsed/60:.1f}m")

df_results = pd.DataFrame(completed_records)
df_results["qa_id"] = df_results["qa_id"].astype(str).str.strip()
df_results = df_gold[["qa_id"]].merge(df_results, on="qa_id", how="left")

assert len(df_results) == 180, f"Error: Output length is {len(df_results)}, expected 180!"
assert df_results["qa_id"].is_unique, "Error: Output contains duplicate qa_id!"
assert set(df_results["qa_id"]) == expected_ids, "Error: Output IDs do not match Gold IDs!"
assert df_results["qwen14b_norag_answer"].notna().all(), "Error: Output contains NaN answers!"
assert (df_results["qwen14b_norag_answer"].str.strip() != "").all(), "Error: Output contains blank answers!"

df_results.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"✓ 180-Question Inference Complete! Saved safely to: {OUTPUT_PATH}")

Starting batch generation for remaining 180/180 questions...
[10/180] Processed (qa_0010) | Elapsed: 7.2m
[20/180] Processed (qa_0020) | Elapsed: 14.3m
[30/180] Processed (qa_0030) | Elapsed: 21.5m
[40/180] Processed (qa_0040) | Elapsed: 28.7m
[50/180] Processed (qa_0050) | Elapsed: 35.9m
[60/180] Processed (qa_0060) | Elapsed: 43.2m
[70/180] Processed (qa_0070) | Elapsed: 50.4m
[80/180] Processed (qa_0080) | Elapsed: 57.7m
[90/180] Processed (qa_0090) | Elapsed: 65.0m
[100/180] Processed (qa_0100) | Elapsed: 72.2m
[110/180] Processed (qa_0110) | Elapsed: 79.3m
[120/180] Processed (qa_0120) | Elapsed: 86.5m
[130/180] Processed (qa_0130) | Elapsed: 93.6m
[140/180] Processed (qa_0140) | Elapsed: 100.7m
[150/180] Processed (qa_0150) | Elapsed: 107.8m
[160/180] Processed (qa_0160) | Elapsed: 115.1m
[170/180] Processed (qa_0170) | Elapsed: 122.3m
[180/180] Processed (qa_0180) | Elapsed: 129.5m
✓ 180-Question Inference Complete! Saved safely to: /kaggle/working/qwen14b_norag_strict_results_1

In [8]:
# Cell 7: Step 3 — Dual Evaluation (Legacy & Multiset F1, Strict vs Legacy Citations) & Export
def normalize_bn(text):
    if pd.isna(text): return ""
    text = unicodedata.normalize("NFC", str(text))
    bn_digits, en_digits = "০১২৩৪৫৬৭৮৯", "0123456789"
    text = text.translate(str.maketrans(bn_digits, en_digits))
    text = re.sub(r"[\[\]\(\)\,\.\-\–\—\:\;\"\'\’]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

def calc_legacy_set_f1(pred, gold):
    p_tokens = set(normalize_bn(pred).split())
    g_tokens = set(normalize_bn(gold).split())
    if not p_tokens or not g_tokens: return 0.0
    common = p_tokens.intersection(g_tokens)
    if not common: return 0.0
    p = len(common) / len(p_tokens)
    r = len(common) / len(g_tokens)
    return (2 * p * r) / (p + r)

def calc_multiset_f1(pred, gold):
    p_tokens = normalize_bn(pred).split()
    g_tokens = normalize_bn(gold).split()
    if not p_tokens or not g_tokens: return 0.0
    common = Counter(p_tokens) & Counter(g_tokens)
    num_same = sum(common.values())
    if num_same == 0: return 0.0
    p = num_same / len(p_tokens)
    r = num_same / len(g_tokens)
    return (2 * p * r) / (p + r)

per_question_rows = []
legacy_act_hits, strict_act_hits = 0, 0
legacy_sec_hits, strict_sec_hits = 0, 0
legacy_f1s, multiset_f1s = [], []

for _, row in df_results.iterrows():
    raw_act = str(row["act_title"]).strip()
    norm_act = normalize_bn(raw_act)
    target_sec = normalize_bn(str(row["section_number"]))
    gold_ans = str(row["final_answer"])
    pred_ans_norm = normalize_bn(row["qwen14b_norag_answer"])
    
    act_core = " ".join(norm_act.split()[:4])
    leg_act_hit = 1 if (act_core in pred_ans_norm or norm_act in pred_ans_norm) else 0
    
    strict_aliases = CANONICAL_ACT_ALIASES.get(raw_act, [norm_act])
    strict_act_hit = 1 if any(normalize_bn(alias) in pred_ans_norm for alias in strict_aliases) else 0
    
    legacy_sec_regex = rf"\b{re.escape(target_sec)}\b"
    leg_sec_hit = 1 if (re.search(legacy_sec_regex, pred_ans_norm) or target_sec in pred_ans_norm) else 0
    
    strict_sec_regex = rf"(?:^|[^\w])(?:ধারা|section|sec)\s*[:\-]?\s*{re.escape(target_sec)}(?:[^\w]|$)"
    strict_sec_hit = 1 if re.search(strict_sec_regex, pred_ans_norm) else 0
    
    leg_f1 = calc_legacy_set_f1(row["qwen14b_norag_answer"], gold_ans)
    multi_f1 = calc_multiset_f1(row["qwen14b_norag_answer"], gold_ans)
    
    legacy_act_hits += leg_act_hit
    strict_act_hits += strict_act_hit
    legacy_sec_hits += leg_sec_hit
    strict_sec_hits += strict_sec_hit
    legacy_f1s.append(leg_f1)
    multiset_f1s.append(multi_f1)
    
    per_question_rows.append({
        "qa_id": row["qa_id"],
        "question": row["question"],
        "gold_act": raw_act,
        "gold_section": target_sec,
        "predicted_answer": row["qwen14b_norag_answer"],
        "legacy_act_match": leg_act_hit,
        "strict_act_match": strict_act_hit,
        "legacy_section_match": leg_sec_hit,
        "strict_section_match": strict_sec_hit,
        "legacy_set_f1": round(leg_f1, 4),
        "multiset_token_overlap_f1": round(multi_f1, 4)
    })

PER_Q_OUTPUT = "/kaggle/working/qwen14b_norag_per_question_evaluation.csv"
pd.DataFrame(per_question_rows).to_csv(PER_Q_OUTPUT, index=False, encoding="utf-8-sig")
print(f"✓ Per-question evaluation saved to: {PER_Q_OUTPUT}")

total = len(df_results)
print("\n" + "="*70)
print("=== FINAL METHODOLOGICAL BENCHMARK: QWEN2.5-14B NO-RAG ===") 
print("="*70)
print(f"Total Evaluated Questions            : {total} (100% verified)")
print(f"Legacy Act Citation Accuracy         : {(legacy_act_hits/total)*100:.2f}% ({legacy_act_hits}/{total})")
print(f"Strict Canonical Act Citation Acc    : {(strict_act_hits/total)*100:.2f}% ({strict_act_hits}/{total})")
print(f"Legacy Section Citation Accuracy     : {(legacy_sec_hits/total)*100:.2f}% ({legacy_sec_hits}/{total})")
print(f"Strict Section Citation Accuracy     : {(strict_sec_hits/total)*100:.2f}% ({strict_sec_hits}/{total})")
print(f"Set-based Token F1 (Legacy Matching) : {sum(legacy_f1s)/total:.4f}")
print(f"Multiset Token-overlap F1            : {sum(multiset_f1s)/total:.4f}")
print("="*70)

eval_summary = pd.DataFrame({
    "Evaluation Metric": [
        "Total Evaluated Questions",
        "Legacy Act Citation Accuracy (%)",
        "Strict Canonical Act Citation Accuracy (%)",
        "Legacy Section Citation Accuracy (%)",
        "Strict Section Citation Accuracy (%)",
        "Set-based Token F1 (Legacy Matching)",
        "Multiset Token-overlap F1"
    ],
    "Qwen2.5-14B No-RAG Value": [
        total,
        f"{(legacy_act_hits/total)*100:.2f}% ({legacy_act_hits}/{total})",
        f"{(strict_act_hits/total)*100:.2f}% ({strict_act_hits}/{total})",
        f"{(legacy_sec_hits/total)*100:.2f}% ({legacy_sec_hits}/{total})",
        f"{(strict_sec_hits/total)*100:.2f}% ({strict_sec_hits}/{total})",
        f"{sum(legacy_f1s)/total:.4f}",
        f"{sum(multiset_f1s)/total:.4f}"
    ]
})
eval_summary.to_csv("/kaggle/working/qwen14b_norag_fully_audited_report.csv", index=False)
print("✓ Aggregate summary exported to /kaggle/working/qwen14b_norag_fully_audited_report.csv")

✓ Per-question evaluation saved to: /kaggle/working/qwen14b_norag_per_question_evaluation.csv

=== FINAL METHODOLOGICAL BENCHMARK: QWEN2.5-14B NO-RAG ===
Total Evaluated Questions            : 180 (100% verified)
Legacy Act Citation Accuracy         : 16.67% (30/180)
Strict Canonical Act Citation Acc    : 16.11% (29/180)
Legacy Section Citation Accuracy     : 14.44% (26/180)
Strict Section Citation Accuracy     : 1.67% (3/180)
Set-based Token F1 (Legacy Matching) : 0.1980
Multiset Token-overlap F1            : 0.1775
✓ Aggregate summary exported to /kaggle/working/qwen14b_norag_fully_audited_report.csv
